In [1]:
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import torch
import timm
from torchvision.io import read_image, ImageReadMode
import torch.nn.functional as F
import numpy as np
import torch.optim as optim
from torch import nn
from collections import Counter
from torchvision.transforms import v2


c:\Users\guilh\Git\Elifoot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def split_data(data_list, train_proportion=0.7, test_proportion=0.2):
    data_list_len = len(data_list)
    shuffled_data = list(np.random.permutation(data_list))

    train_size = int(np.ceil(data_list_len * train_proportion))
    test_size = int(np.ceil(data_list_len * test_proportion))
    
    return {'train': shuffled_data[0: train_size], 
            'test': shuffled_data[train_size: train_size + test_size], 
            'val': shuffled_data[train_size + test_size:]}


In [3]:
class EliImageDataset(Dataset):
    def __init__(self, img_path_list, all_classes_df=None, transform=None):

        self.img_path_list = img_path_list
        self.img_path_df = pd.Series(self.img_path_list).explode().reset_index()
        self.img_path_df.columns = ['Label', 'Img_path'] 

        if all_classes_df is not None:
            self.all_classes_df = all_classes_df
        else:
            cd = Counter([i.split('\\')[-2] for i in self.img_path_list])
            classes_df = pd.DataFrame(cd, index=[0]).transpose().reset_index()
            classes_df.columns = ['Class', 'Count']
            classes_df['Label'] = classes_df.index
            self.all_classes_df = classes_df

        if not transform:
            self.transform = v2.Compose([
                v2.ToImage(),             
                v2.ToDtype(torch.float32, scale=True),
                v2.Resize((120, 120), antialias=True)
                ])
        else:
            self.transform = transform


    def __len__(self):
        return len(self.img_path_list)

    def __getitem__(self, idx):
        image = read_image(self.img_path_df.iloc[idx]['Img_path'], mode=ImageReadMode.RGB)
        label = self.img_path_df.iloc[idx]['Img_path'].split('\\')[2]

        if self.transform:
            image = self.transform(image)

        label = self.all_classes_df.loc[self.all_classes_df['Class'] == label, 'Label'].item()
        return image, label

In [4]:
img_labels_folder = r'Data\Frames_Categories'
img_path_dict = {}
for folder in os.listdir(img_labels_folder):
    if folder == 'Other':
        continue
    folder_path = os.path.join(img_labels_folder, folder)
    img_path_dict[folder] =[os.path.join(folder_path, img_name) for img_name in  os.listdir(folder_path)]

In [5]:
imgs_split_dict = {split_type: [] for split_type in ['train', 'test', 'val']}

for img_class in img_path_dict.keys():
    if len(img_path_dict[img_class]) < 0:
        continue
    class_split_dict = split_data(img_path_dict[img_class])
    for split_type in class_split_dict.keys(): 
        imgs_split_dict[split_type] += class_split_dict[split_type]

In [6]:
transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    v2.RandomResizedCrop(size=(120, 120), antialias=True),
])

In [7]:
ds_train = EliImageDataset(imgs_split_dict['train'], transform=transforms)
ds_val = EliImageDataset(imgs_split_dict['val'], ds_train.all_classes_df)
ds_test = EliImageDataset(imgs_split_dict['test'], ds_train.all_classes_df)

In [8]:
train_dataloader = DataLoader(ds_train, batch_size=64, shuffle=True)
val_dataloader = DataLoader(ds_val, batch_size=64)
test_dataloader = DataLoader(ds_test, batch_size=64)

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
w = torch.tensor(ds_train.all_classes_df['Count'] / ds_train.all_classes_df['Count'].sum()).float()

In [11]:
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=ds_train.all_classes_df.shape[0])

criterion = torch.nn.CrossEntropyLoss()#1/w
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='max',
                                                       patience=1, factor=0.4)
softmax = nn.Softmax(dim=1)

In [13]:
all_losses = []
d = {}
train_outputs_d = {}

for epoch in range(50):
    epoch_train_loss = []
    epoch_preds = []
    epoch_labels = []
    train_outputs_d[epoch] = {}
    counter = 0
    for img, labels in train_dataloader:

        outputs = model.to(device)(img.to(device))
        loss = criterion(outputs, labels.to(device))

        preds = softmax(outputs).max(axis=1).indices.tolist()
        epoch_train_loss += [loss.item()]

        optimizer.zero_grad()


        loss.backward()
        optimizer.step()

        epoch_outputs = preds
        epoch_labels += [label.item() for label in labels]
        #Accuracy

        train_outputs_d[epoch][counter] = outputs
        counter += 1
    
    epoch_mean_loss_train = np.mean(epoch_train_loss)
    d[epoch] = pd.DataFrame([epoch_preds, epoch_labels]).transpose()
 
    all_preds_val = []
    all_labels_val = []

    for img_val, label_val in test_dataloader:
        with torch.no_grad():
            outputs = model.to(device)(img_val)
            all_preds_val += softmax(outputs).max(axis=1).indices.tolist()
            all_labels_val += label_val.tolist()

    preds_df = pd.DataFrame([all_preds_val, all_labels_val]).transpose()
    preds_df.columns = ['Pred', 'Label']
    
    epoch_mean_loss = np.mean(preds_df['Pred'] == preds_df['Label'])
    print(f'Epoch {epoch}, Train:{epoch_mean_loss_train}, Val accuracy:{epoch_mean_loss}')
 


Epoch 0, Train:2.320320467154185, Val accuracy:0.5887850467289719
Epoch 1, Train:1.0065742234388988, Val accuracy:0.8037383177570093
Epoch 2, Train:0.5288664350907007, Val accuracy:0.8598130841121495
Epoch 3, Train:0.36675908664862317, Val accuracy:0.7850467289719626
Epoch 4, Train:0.3182177394628525, Val accuracy:0.9065420560747663
Epoch 5, Train:0.22274079049626985, Val accuracy:0.8691588785046729
Epoch 6, Train:0.22283333291610083, Val accuracy:0.9158878504672897
Epoch 7, Train:0.1457155222694079, Val accuracy:0.8878504672897196
Epoch 8, Train:0.1989877906938394, Val accuracy:0.9439252336448598
Epoch 9, Train:0.22272306804855665, Val accuracy:0.8691588785046729
Epoch 10, Train:0.22247306754191717, Val accuracy:0.6074766355140186
Epoch 11, Train:0.19711374615629515, Val accuracy:0.9065420560747663
Epoch 12, Train:0.14640950535734495, Val accuracy:0.9813084112149533
Epoch 13, Train:0.1253526924798886, Val accuracy:0.9065420560747663
Epoch 14, Train:0.13923920939366022, Val accuracy:0.

In [14]:
torch.save(model, 'Models/frame_model.pt')
ds_train.all_classes_df.to_csv('classes_label_conversion.csv', index=False)

In [ ]:
1/0

## Model usage

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torch.load('Models/frame_model.pt', weights_only=False)

In [ ]:
all_preds = []
all_labels = []

softmax = nn.Softmax(dim=1)
for img, label in test_dataloader:
    outputs = model.to(device)(img)
    all_preds += softmax(outputs).max(axis=1).indices.tolist()
    all_labels += label.tolist()
    

In [ ]:
np.mean(np.array(all_preds) == np.array(all_labels))

np.float64(0.9813084112149533)